# Praktikum 2 - Klasifikasi Multikelas JST dengan Keras pada Dataset PBL

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
import tensorflow as tf

# Load dataset PBL
df = pd.read_csv('burnout_dataset1.csv')
df_sample, _ = train_test_split(
    df, train_size=5000, stratify=df['Burnout_Risk'], random_state=42
)

num_cols = df_sample.select_dtypes(include=[np.number]).columns.tolist()
feature_cols = [c for c in num_cols if c not in ['Person_ID', 'Burnout_Score']]

X_raw = SimpleImputer(strategy='median').fit_transform(df_sample[feature_cols])
X = StandardScaler().fit_transform(X_raw)
y_raw = df_sample[['Burnout_Risk']]

# One-hot encoding target (Low, Moderate, High)
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y_raw)
print("Kelas target:", encoder.categories_[0])

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=df_sample['Burnout_Risk'], random_state=42
)

# Bangun model
tf.keras.utils.set_random_seed(42)
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])

# Kompilasi
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Latih model
model.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)

# Evaluasi
loss, acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Loss Awal (10, 8 - ReLU): {loss:.4f}, Akurasi: {acc:.4f}")

loss_awal, acc_awal = loss, acc

Kelas target: ['High' 'Low' 'Moderate']


Loss Awal (10, 8 - ReLU): 0.4290, Akurasi: 0.7830


# Tugas 2 - Variasi Jumlah Neuron Hidden Layer

In [2]:
# Variasi 1: Hidden layer lebih kecil (5, 3)
tf.keras.utils.set_random_seed(42)
model_var1 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(5, activation='relu'),
    tf.keras.layers.Dense(3, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
model_var1.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_var1.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_v1, acc_v1 = model_var1.evaluate(X_test, y_test, verbose=0)

# Variasi 2: Hidden layer lebih besar (32, 16)
tf.keras.utils.set_random_seed(42)
model_var2 = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
model_var2.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_var2.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_v2, acc_v2 = model_var2.evaluate(X_test, y_test, verbose=0)

print(f"Akurasi Model Awal (10, 8): {acc_awal:.4f}")
print(f"Akurasi Model Variasi 1 (5, 3): {acc_v1:.4f}")
print(f"Akurasi Model Variasi 2 (32, 16): {acc_v2:.4f}")

Akurasi Model Awal (10, 8): 0.7830
Akurasi Model Variasi 1 (5, 3): 0.8010
Akurasi Model Variasi 2 (32, 16): 0.7800


# Tugas 3 - Perbandingan Fungsi Aktivasi Sigmoid vs ReLU

In [3]:
# Model Sigmoid (10, 8)
tf.keras.utils.set_random_seed(42)
model_sigmoid = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(10, activation='sigmoid'),
    tf.keras.layers.Dense(8, activation='sigmoid'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
model_sigmoid.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_sigmoid.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_sig, acc_sig = model_sigmoid.evaluate(X_test, y_test, verbose=0)

# Model ReLU (10, 8)
tf.keras.utils.set_random_seed(42)
model_relu = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(10, activation='relu'),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax'),
])
model_relu.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_relu.fit(X_train, y_train, epochs=50, batch_size=32, verbose=0)
loss_relu_eval, acc_relu_eval = model_relu.evaluate(X_test, y_test, verbose=0)

df_eval = pd.DataFrame({
    'Model': [
        'Model Awal (ReLU)',
        'Variasi 1 (Kecil - ReLU)',
        'Variasi 2 (Besar - ReLU)',
        'Model Sigmoid',
        'Model ReLU (Pembanding)',
    ],
    'Neuron Hidden': ['[10, 8]', '[5, 3]', '[32, 16]', '[10, 8]', '[10, 8]'],
    'Aktivasi': ['ReLU', 'ReLU', 'ReLU', 'Sigmoid', 'ReLU'],
    'Loss Uji': [round(loss_awal, 4), round(loss_v1, 4), round(loss_v2, 4), round(loss_sig, 4), round(loss_relu_eval, 4)],
    'Akurasi Uji': [round(acc_awal, 4), round(acc_v1, 4), round(acc_v2, 4), round(acc_sig, 4), round(acc_relu_eval, 4)],
})
df_eval

,Model,Neuron Hidden,Aktivasi,Loss Uji,Akurasi Uji
0,Model Awal (ReLU),"[10, 8]",ReLU,0.4290,0.783
1,Variasi 1 (Kecil - ReLU),"[5, 3]",ReLU,0.3991,0.801
2,Variasi 2 (Besar - ReLU),"[32, 16]",ReLU,0.4766,0.780
3,Model Sigmoid,"[10, 8]",Sigmoid,0.4160,0.792
4,Model ReLU (Pembanding),"[10, 8]",ReLU,0.4290,0.783


Berdasarkan hasil pengujian klasifikasi tiga kelas Burnout_Risk (Low, Moderate, High) pada dataset PBL:

1. Variasi jumlah neuron hidden layer menunjukkan pengaruh kapasitas model terhadap akurasi uji. Konfigurasi kecil [5, 3] tetap dapat mempelajari pola utama, sedangkan konfigurasi lebih besar [32, 16] memiliki kapasitas representasi lebih tinggi namun perlu diperhatikan agar tidak mengalami overfitting pada data latih.
2. Pada perbandingan fungsi aktivasi dengan arsitektur [10, 8] selama 50 epoch, ReLU dan Sigmoid sama-sama mampu mengklasifikasikan data tabular terstandarisasi, dengan perbedaan kecepatan konvergensi dan nilai loss akhir yang terlihat pada tabel evaluasi.